# Readme Vorbereitung gocfl create Befehle

Dieses Jupyiter Notebook erstellt die gocfl create-Befehle für eine bestimmte Collection. 
Die Doku für den Aufbau eines gocfl create Befehls befindet sich hier: https://github.com/je4/gocfl/blob/main/docs/create.md

## config.py

In der Config wird die aktuell zu verarbeitende Collection sowie diverse Dateipfade konfiguriert. Bspw. für E-Rara:

    collection = 'e-rara'
    root = 'zhb_archiv/e-rara'
    sigpath = 'fulldump/signatures.txt'

## signature

Die Signature ist zentral für die Erstellung des storage roots, sowie das Auffinden der Objekt-Pfade, Metadaten und Info-Dateien. Grundsätzlich sollte für jede Collection eine Textdatei namens '/signatures.txt' mit den signatures vorliegen. Diese werden entweder durch ein anderes Jupyter Notebook konfiguriert oder können von Hand erstellt werden.
Der Dateipfad kann konfiguriert werden. 

## storage root

Hier wird davon ausgegangen, dass der storage root ein ZIP file sein soll. Für jede signature wird ein storage_root angelegt. Der Root-Path kann konfiguriert werden. 
Fürs Testen muss nach dem Durchlauf des Scripts der storage root Ordner wieder geleert werden, ansonsten geht das script davon aus, dass das Archiv bereits besteht und ein Update notwendig ist. 


###  Create Befehl für gocfl generieren

Die gocfl create Befehle für alle Signaturen werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt. Der Pfad für die Config.toml kann konfiguriert werden.  

Muster:

    gocfl create ./archiv.zip ./object-directory metadata:./metadata-directory --config ./config/gocfl.toml -i 'signature'  --ext-NNNN-metafile-source ./info.json


In [5]:
import config
import os
from zipfile import ZipFile
import shutil
from datetime import datetime

#prepare archive structure

ingest = config.ingest
root = config.root
archive = config.archive
collection = config.collection
sig_path = config.sigpath
sig_file = f'{ingest}/{sig_path}/signatures.txt'
md_path =  config.mdpath
md_format = config.mdformat
info_path = config.infopath
obj_path = config.objpath
gocfl_config = config.configfile

# create archive root
if os.path.exists(archive):
    print(f"Existing ZHB Archive directory: {archive}")    

else:
    os.mkdir(archive)
    print(f"ZHB Archive created: {archive}")

# create archive collection
collection_path =  os.path.join(archive, collection)

if os.path.exists(collection_path):
        print(f"Existing collection directory: {collection_path}")
else:
    os.mkdir(collection_path)
    print(f"Collection directory created: {collection_path}")

# remove old directories, create new ones
if os.path.exists('gocfl_create'):
    shutil.rmtree('gocfl_create')
    
if os.path.exists('gocfl_update'):
    shutil.rmtree('gocfl_update')    

if os.path.exists('gocfl_errors'):
    shutil.rmtree('gocfl_errors')   
    
os.mkdir('gocfl_update')
os.mkdir('gocfl_create')
os.mkdir('gocfl_errors')

# read line from signaturesfile
with open(sig_file, 'r') as file:
    
    print("\nOpening file:",sig_file, "\n")
    for row in file:
        signature = row.strip()
        print("Signature:",signature)
        folder = signature[4:]
        
        # create filepaths for metadata, info.json, objects:
        metadata_folder = f'{ingest}/{md_path}/{folder}'
        print("           Metadata folder: ",metadata_folder)
        info_file = f'{ingest}/{info_path}/{signature}.json'        
        print("           Info.json: ",info_file)
        object_path = f'{ingest}/{obj_path}/{folder}'
        print("           Object path: ",object_path)
        
        # check for empty object path
        if len(os.listdir(object_path)) == 0: 
            
            print(f"!!!!!!!!!! Storage root {storage_root} is empty, check for errors or missing files. ") 
            with open(f'gocfl_errors/error_{signature}.txt', 'w') as file:
                file.write(signature)
                print(f"!!!!!!!!!! Signature added to 'gocfl_errors/error_{signature}.txt'")
            continue 
        
        # create storage root for this object
        storage_root = f'{archive}/{collection}/{signature}.zip'
            
        if os.path.exists(storage_root):
            # update procedure necessary, write signature to update file and continue
            print("!!!!!!!!!! Storage root exists already, use gocfl update!")
            with open(f'gocfl_update/gocfl_update_{signature}.txt', 'w') as file:
                file.write(signature)
                print(f"!!!!!!!!!! Signature added to 'update/ocfl_update_{signature}.txt'")
            continue    

        else:
            with ZipFile(storage_root, 'w') as zipfile:
                print("           Storage root created: ",storage_root)
            
            # create string
            create_string = f'gocfl create {root}/{storage_root} {root}/{object_path} metadata:{root}/{metadata_folder} --config {root}/{gocfl_config} -i "{signature}"  --ext-NNNN-metafile-source {root}/{info_file}'
            print('\n###############\n',create_string, '\n###############\n')
            create_file = f'gocfl_create/gocfl_create_{signature}.txt'
            with open(create_file, 'w') as file:
                file.write(create_string)
                
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))                

Existing ZHB Archive directory: zhb_dlza
Existing collection directory: zhb_dlza\sosa_e-manuscripta

Opening file: e-manuscripta/files/signatures.txt 

Signature: zhb_10_7891_e-manuscripta-108732
           Metadata folder:  e-manuscripta/metadata/10_7891_e-manuscripta-108732
           Info.json:  e-manuscripta/info/zhb_10_7891_e-manuscripta-108732.json
           Object path:  e-manuscripta/objects/10_7891_e-manuscripta-108732
           Storage root created:  zhb_dlza/sosa_e-manuscripta/zhb_10_7891_e-manuscripta-108732.zip

###############
 gocfl create ./zhb_dlza/sosa_e-manuscripta/zhb_10_7891_e-manuscripta-108732.zip ./e-manuscripta/objects/10_7891_e-manuscripta-108732 metadata:./e-manuscripta/metadata/10_7891_e-manuscripta-108732 --config ./config/gocfl.toml -i "zhb_10_7891_e-manuscripta-108732"  --ext-NNNN-metafile-source ./e-manuscripta/info/zhb_10_7891_e-manuscripta-108732.json 
###############

Signature: zhb_10_7891_e-manuscripta-24104
           Metadata folder:  e-manuscri